In [26]:
import numpy as np
import pandas as pd
import scipy.stats as stats

In [27]:
EXERCISE = 1
df = pd.read_csv(f'./data/Exercise_{EXERCISE}_valid.csv')
df.ID = df.ID.astype(int)
df.TrialInfo = df.TrialInfo.astype(int)
df.Repetitions = df.Repetitions.astype(float)

In [28]:
df_mean = df.groupby(["ID", "TrialInfo"]).agg(
    Mean_Repetitions=("Repetitions", "mean"),
    Group=("Group", "first")
).reset_index()

#sort by ID and TrialInfo
df_mean = df_mean.sort_values(by=["ID", "TrialInfo"])

In [29]:
# Only keep TrialInfo 1 and 4
df_filtered = df_mean[df_mean['TrialInfo'].isin([1, 4])]

# Pivot to wide format: one row = one student
df_pivot = df_filtered.pivot_table(index=['ID', 'Group'],
                                   columns='TrialInfo',
                                   values='Mean_Repetitions').reset_index()

# Rename columns for easier handling
df_pivot = df_pivot.rename(columns={1: 'Week1', 4: 'Week4'})

print(df_pivot.head())

TrialInfo  ID Group  Week1  Week4
0           1    VR    1.8    6.6
1           2    VR    1.0    5.2
2           4    VR    2.4    1.4
3           6    VR    2.2    4.4
4           8    VR    1.8    2.0


# Step 2: Check normality within each group

We check if the differences across weeks are roughly normal (important for Repeated Measures ANOVA).

In [30]:
results = {}

for group in df_pivot['Group'].unique():
    # Subset the group
    group_data = df_pivot[df_pivot['Group'] == group]

    week1 = group_data['Week1']
    week4 = group_data['Week4']

    # Check normality of the differences
    stat, p_normality = stats.shapiro(week1 - week4)
    print(f"\nGroup: {group}")
    print(f"Normality test p-value: {p_normality:.4f}")

    if p_normality > 0.05:
        # Differences are normal --> Paired t-test
        t_stat, p_value = stats.ttest_rel(week1, week4)
        test_used = 'Paired t-test'
    else:
        # Differences are not normal --> Wilcoxon signed-rank test
        w_stat, p_value = stats.wilcoxon(week1, week4)
        test_used = 'Wilcoxon signed-rank'

    # Determine if significant
    significant = p_value < 0.05
    significance_text = "✅ Significant difference!" if significant else "❌ No significant difference."

    # Save results
    results[group] = {
        'test': test_used,
        'p_value': p_value,
        'significant': significant,
        'text': significance_text
    }

# Show final results
print("\n--- Final Results ---")
for group, res in results.items():
    print(f"\nGroup: {group}")
    print(f"Test used: {res['test']}")
    print(f"p-value: {res['p_value']:.4f}")
    print(f"Result: {res['text']}")



Group: VR
Normality test p-value: 0.0001

Group: RL
Normality test p-value: 0.0517

Group: Kontrollgruppe
Normality test p-value: 0.3445

--- Final Results ---

Group: VR
Test used: Wilcoxon signed-rank
p-value: 0.0013
Result: ✅ Significant difference!

Group: RL
Test used: Paired t-test
p-value: 0.0000
Result: ✅ Significant difference!

Group: Kontrollgruppe
Test used: Paired t-test
p-value: 0.0735
Result: ❌ No significant difference.


# Step 3: Across all groups

In [31]:
# Pivot to wide format: one row = one student
df_pivot = df_mean.pivot_table(index=['ID', 'Group'],
                               columns='TrialInfo',
                               values='Mean_Repetitions').reset_index()

# Rename columns for easier handling
df_pivot = df_pivot.rename(columns={1: "Week1", 2: "Week2", 3: "Week3", 4: 'Week4'})

print(df_pivot)

TrialInfo   ID           Group  Week1  Week2  Week3  Week4
0            1              VR    1.8    2.2    2.8    6.6
1            2              VR    1.0    1.4    2.2    5.2
2            4              VR    2.4    1.4    1.8    1.4
3            6              VR    2.2    1.6    5.4    4.4
4            8              VR    1.8    2.0    2.2    2.0
..         ...             ...    ...    ...    ...    ...
84         139  Kontrollgruppe   98.0  100.0   83.6  100.0
85         141  Kontrollgruppe   48.6   60.6   59.4   56.2
86         143  Kontrollgruppe    5.8    3.0    5.0    5.0
87         144  Kontrollgruppe   73.0  100.0  100.0  100.0
88         148  Kontrollgruppe    7.6    8.4    9.2   12.8

[89 rows x 6 columns]


In [32]:
from sklearn.linear_model import LinearRegression

# Define X once, same for all
X = np.array([1, 2, 3, 4]).reshape(-1, 1)


# Function to calculate slope for one student
def calculate_slope(row):
    y = np.array([row['Week1'], row['Week2'], row['Week3'], row['Week4']])
    model = LinearRegression().fit(X, y)
    return model.coef_[0]  # coef_ returns an array, take first element (slope)


# Apply to each row
df_pivot['Slope'] = df_pivot.apply(calculate_slope, axis=1)

# Add metrics
df_pivot['Improvement_%'] = (df_pivot['Week4'] - df_pivot['Week1']) / df_pivot['Week1'] * 100


In [33]:
import scipy.stats as stats

# Define all pairs you want to compare
group_pairs = [('VR', 'Kontrollgruppe'), ('RL', 'Kontrollgruppe'), ('VR', 'RL')]

# Metrics to analyze
metrics = ['Improvement_%', 'Slope']

results = {}

for metric in metrics:
    print(f"\n\n--- Analyzing {metric} ---")
    results[metric] = {}

    for group1, group2 in group_pairs:
        data1 = df_pivot[df_pivot['Group'] == group1][metric]
        data2 = df_pivot[df_pivot['Group'] == group2][metric]

        data1 = data1.replace([np.inf, -np.inf], np.nan).dropna()
        data2 = data2.replace([np.inf, -np.inf], np.nan).dropna()

        # Check normality for both groups
        p_norm1 = stats.shapiro(data1)[1]
        p_norm2 = stats.shapiro(data2)[1]

        normal = (p_norm1 > 0.05) and (p_norm2 > 0.05)

        if normal:
            # Use independent t-test
            stat, p_value = stats.ttest_ind(data1, data2)
            test_used = 'Independent t-test'
        else:
            # Use Mann-Whitney U test
            stat, p_value = stats.mannwhitneyu(data1, data2, alternative='two-sided')
            test_used = 'Mann-Whitney U'

        # Save results
        results[metric][(group1, group2)] = {
            'test': test_used,
            'p_value': p_value
        }

        # Print results
        print(f"\n{group1} vs {group2}")
        print(f"Test used: {test_used}")
        print(f"p-value: {p_value:.4f}")
        if p_value < 0.05:
            print("✅ Significant difference!")
        else:
            print("❌ No significant difference.")




--- Analyzing Improvement_% ---

VR vs Kontrollgruppe
Test used: Mann-Whitney U
p-value: 0.1251
❌ No significant difference.

RL vs Kontrollgruppe
Test used: Mann-Whitney U
p-value: 0.0008
✅ Significant difference!

VR vs RL
Test used: Mann-Whitney U
p-value: 0.1813
❌ No significant difference.


--- Analyzing Slope ---

VR vs Kontrollgruppe
Test used: Mann-Whitney U
p-value: 0.5681
❌ No significant difference.

RL vs Kontrollgruppe
Test used: Mann-Whitney U
p-value: 0.0009
✅ Significant difference!

VR vs RL
Test used: Mann-Whitney U
p-value: 0.0000
✅ Significant difference!


# Table

In [34]:
import pandas as pd
import numpy as np

# Metrics you want to report
metrics = ['Improvement_%', 'Slope']

# Create empty list to collect rows
summary_data = []

# Loop through each group
for group in df_pivot['Group'].unique():
    row = {'Gruppe': group}

    for metric in metrics:
        group_data = df_pivot[df_pivot['Group'] == group][metric]

        # Remove inf, -inf, and NaN before calculation
        group_data_clean = group_data.replace([np.inf, -np.inf], np.nan).dropna()

        mean = group_data_clean.mean()
        std = group_data_clean.std()

        # Format "mean ± std"
        row[metric] = f"{mean:.2f} ± {std:.2f}"

    # Add number of participants (after cleaning inf values if you want)
    n_participants = df_pivot[df_pivot['Group'] == group]['ID'].nunique()
    row['Anzahl der Teilnehmenden'] = n_participants

    summary_data.append(row)

# Create DataFrame
summary_df = pd.DataFrame(summary_data)

# Reorder columns nicely
summary_df = summary_df[['Gruppe', 'Improvement_%', 'Slope', 'Anzahl der Teilnehmenden']]

# Rename columns for nice output
summary_df.columns = ['Gruppe', 'Metrik 1 (Improvement %)', 'Metrik 2 (Slope)', 'Anzahl der Teilnehmenden']

# Show final table
print(summary_df)

# save to ./results as csv
summary_df.to_csv(f'./results/summary_table_{EXERCISE}.csv', index=False)
df_pivot.to_csv(f'./results/df_metrics_{EXERCISE}.csv', index=False)


           Gruppe Metrik 1 (Improvement %) Metrik 2 (Slope)  \
0              VR          108.20 ± 142.28      0.79 ± 1.68   
1              RL          135.32 ± 114.79      8.12 ± 6.27   
2  Kontrollgruppe            28.90 ± 58.70      2.29 ± 4.09   

   Anzahl der Teilnehmenden  
0                        30  
1                        47  
2                        12  
